# 20 — Competitive feature-forecasting research

This notebook is the employer-facing analysis chapter for Round 20. It evaluates independently implemented mechanisms from strong public MITSUI solutions under the repository's existing point-in-time validation contract.

**Important:** this notebook does not evaluate the reserved final origins. Run the terminal commands in the accompanying protocol first, then return here to inspect results and figures.


## Research questions

1. Does recursive forecasting of target-linked underlying market series transfer to our development protocol?
2. Does horizon-correct released-label correction add signal beyond the LSTM?
3. Does the public covariance/rank prior add stable cross-sectional signal?
4. Are the public-inspired models complementary enough to improve the canonical `current_market` system when blended?


In [1]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
print("Project root:", ROOT)
latest_path = ROOT / "artifacts/competitive_feature_forecast/LATEST.json"
assert latest_path.exists(), "Run the smoke/first-fold experiment before this notebook."
latest = json.loads(latest_path.read_text())
summary = json.loads((ROOT / latest["summary"]).read_text())
print(json.dumps({k: summary[k] for k in ["status", "run_id", "mode", "development_dates", "reserved_final_origins_evaluated"]}, indent=2))


Project root: /home/sagemaker-user/projects/commodity-prediction-current
{
  "status": "completed",
  "run_id": "b1cd1ddebb99b530",
  "mode": "smoke",
  "development_dates": 180,
  "reserved_final_origins_evaluated": false
}


## Score comparison

The official metric is the mean daily Spearman rank correlation divided by its population standard deviation. Scores below are local development scores, not Kaggle leaderboard scores.


In [2]:
rows = []
if summary["pooled_results"]:
    for row in summary["pooled_results"]:
        rows.append({
            "variant": row["variant"],
            "official_metric": row["official_metric"],
            "mean_daily_rank_correlation": row["mean_daily_rank_correlation"],
            "positive_correlation_fraction": row["positive_correlation_fraction"],
        })
else:
    # first-fold mode still has pooled_results over the single completed fold
    rows = [{
        "variant": row["variant"],
        "official_metric": row["official_metric"],
        "mean_daily_rank_correlation": row["mean_daily_rank_correlation"],
        "positive_correlation_fraction": row["positive_correlation_fraction"],
    } for row in summary["fold_results"] if row["fold"] == 0]
scores = pd.DataFrame(rows).drop_duplicates("variant").sort_values("official_metric", ascending=False)
scores


,variant,official_metric,mean_daily_rank_correlation,positive_correlation_fraction
0,current_market,0.403381,0.070685,0.655556
1,current_market_plus_kelly_prior,0.167708,0.005835,0.555556
2,mean_rank_prior,0.165663,0.029020,0.566667
3,regularized_kelly_rank_prior,0.156027,0.005438,0.544444
4,lstm_feature_forecast_log,-0.057828,-0.011237,0.511111
5,current_market_plus_lstm_feature_forecast_log_...,-0.084940,-0.016814,0.483333
6,lstm_feature_forecast_log_released_correction,-0.107909,-0.021483,0.466667
7,released_mean_5,-0.167003,-0.047100,0.455556


In [3]:
fig = px.bar(
    scores.sort_values("official_metric"),
    x="official_metric", y="variant", orientation="h",
    title=f"Round 20 official metric — {summary['mode']} ({summary['development_dates']} development dates)",
    hover_data=["mean_daily_rank_correlation", "positive_correlation_fraction"],
)
fig.add_vline(x=summary["canonical_reference"]["expected_535_date_metric"], line_dash="dash", annotation_text="canonical 535-date current_market reference")
fig.update_layout(height=max(500, 34 * len(scores) + 140), xaxis_title="Official metric (mean daily Spearman / population std)", yaxis_title="Variant")
fig.show()


## Fold-level stability

A candidate is more credible when it helps across time rather than winning from one volatile interval.


In [4]:
fold_df = pd.DataFrame([{
    "fold": row["fold"],
    "variant": row["variant"],
    "official_metric": row["official_metric"],
    "delta_vs_current_market": row.get("delta_vs_current_market"),
} for row in summary["fold_results"]])
fold_df.sort_values(["fold", "official_metric"], ascending=[True, False]).head(30)


,fold,variant,official_metric,delta_vs_current_market
0,0,current_market,0.403381,NaN
7,0,current_market_plus_kelly_prior,0.167708,-0.235673
1,0,mean_rank_prior,0.165663,-0.237718
2,0,regularized_kelly_rank_prior,0.156027,-0.247354
4,0,lstm_feature_forecast_log,-0.057828,-0.461209
6,0,current_market_plus_lstm_feature_forecast_log_...,-0.084940,-0.488321
5,0,lstm_feature_forecast_log_released_correction,-0.107909,-0.511290
3,0,released_mean_5,-0.167003,-0.570384


In [5]:
fig = px.line(
    fold_df, x="fold", y="official_metric", color="variant", markers=True,
    title="Official metric by chronological fold",
)
fig.update_layout(xaxis_title="Chronological fold", yaxis_title="Official metric", height=600)
fig.show()


## Daily rank-correlation path

The cumulative mean below helps distinguish persistent signal from a short lucky burst.


In [6]:
chosen = scores.iloc[0]["variant"]
base = "current_market"
records = []
for row in summary["fold_results"]:
    if row["variant"] not in {chosen, base}:
        continue
    daily = row["daily_rank_correlations"]
    for i, value in enumerate(daily):
        records.append({"fold": row["fold"], "step": i, "variant": row["variant"], "daily_rank_correlation": value})
daily_df = pd.DataFrame(records)
daily_df["global_step"] = daily_df.groupby("variant").cumcount()
daily_df["cumulative_mean"] = daily_df.groupby("variant")["daily_rank_correlation"].expanding().mean().reset_index(level=0, drop=True)
fig = px.line(daily_df, x="global_step", y="cumulative_mean", color="variant", title=f"Cumulative mean daily rank correlation: {chosen} vs current_market")
fig.update_layout(xaxis_title="Development prediction step", yaxis_title="Cumulative mean daily Spearman", height=500)
fig.show()


## Training diagnostics

For LSTM variants, inspect the selected epoch and validation-loss curve. A much earlier optimum than 25 epochs is evidence that the public default should not be copied mechanically.


In [7]:
training_rows=[]
for row in summary["fold_results"]:
    training=row.get("training")
    if not training:
        continue
    for point in training["selection_history"]:
        training_rows.append({"fold": row["fold"], "variant": row["variant"], **point, "best_epoch": training["best_epoch"]})
training_df=pd.DataFrame(training_rows)
training_df.head()


,fold,variant,epoch,train_loss,valid_loss,best_epoch
0,0,lstm_feature_forecast_log,1,0.847568,2.435668,3
1,0,lstm_feature_forecast_log,2,0.492441,1.841068,3
2,0,lstm_feature_forecast_log,3,0.273410,1.616660,3


In [8]:
if not training_df.empty:
    melted = training_df.melt(id_vars=["fold","variant","epoch","best_epoch"], value_vars=["train_loss","valid_loss"], var_name="loss_type", value_name="loss")
    fig=px.line(melted, x="epoch", y="loss", color="variant", line_dash="loss_type", facet_row="fold", markers=True, title="LSTM model-selection loss curves")
    fig.update_layout(height=max(500, 320 * training_df["fold"].nunique()))
    fig.show()


## Decision record

Do **not** promote a model because it resembles a high-ranking public solution. Promote it only if our leakage-safe development evidence supports it.

After saving this notebook, package the return evidence with:

```bash
python scripts/run_competitive_feature_forecast.py --package-return
```

Return `competitive_feature_forecast_return.zip` to ChatGPT. The next gate will be selected from the evidence: full 535-date confirmation, 3rd-place target-pair tree stacking, or a stop/pivot if the mechanism does not transfer.
